In [9]:
### ===== Tahoe-100M: can we repeat the well-level tests? =====
### (step 1: metadata only, no count matrices)
### Run on your machine (needs internet). pip install huggingface_hub pyarrow pandas
### Question: how many DMSO wells ("sample" = one well) does each plate have?
### Result: 2 DMSO wells per plate (28 in total), enough for the tests.
### Cells per cell line in each DMSO well are counted in step 2 (tahoe_dmso_pseudobulk).

import pandas as pd, pyarrow.parquet as pq
from huggingface_hub import HfFileSystem, list_repo_files

REPO = "tahoebio/Tahoe-100M"
files = list_repo_files(REPO, repo_type="dataset")
meta_files = [f for f in files if f.startswith("metadata/")]
data_files = sorted(f for f in files if f.startswith("data/") and f.endswith(".parquet"))
print("metadata files:", meta_files)
print("data files:", len(data_files))

fs = HfFileSystem()

metadata files: ['metadata/cell_line_metadata.parquet', 'metadata/drug_metadata.parquet', 'metadata/gene_metadata.parquet', 'metadata/gene_vocabulary.json', 'metadata/gene_vocabulary.jsonl', 'metadata/obs_metadata.parquet', 'metadata/pseudobulk_differential_expression/train-00000-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00001-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00002-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00003-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00004-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00005-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00006-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00007-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00008-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00009-of-01026.parquet', 'metadata/pseudo

In [10]:
# 1. sample-level metadata, if provided
for f in meta_files:
    if "sample" in f.lower():
        sm = pd.read_parquet(f"hf://datasets/{REPO}/{f}", filesystem=fs)
        print(f, sm.shape); print(sm.head().to_string()); print(list(sm.columns))

metadata/sample_metadata.parquet (1344, 8)
     sample   plate  mean_gene_count  mean_tscp_count  mean_mread_count  mean_pcnt_mito          drug               drugname_drugconc
0  smp_1495  plate1      1354.169768      2027.115940       2444.032416        0.033956  Infigratinib  [('Infigratinib', 0.05, 'uM')]
1  smp_1496  plate1      1404.454157      2226.282791       2690.685970        0.071723  Erdafitinib   [('Erdafitinib ', 0.05, 'uM')]
2  smp_1497  plate1      1205.267794      1859.375821       2246.200127        0.084853    Everolimus    [('Everolimus', 0.05, 'uM')]
3  smp_1498  plate1      1225.510822      1906.494566       2298.907623        0.088262   Pemigatinib   [('Pemigatinib', 0.05, 'uM')]
4  smp_1499  plate1      1231.372881      1861.305085       2245.372881        0.050802   Abemaciclib   [('Abemaciclib', 0.05, 'uM')]
['sample', 'plate', 'mean_gene_count', 'mean_tscp_count', 'mean_mread_count', 'mean_pcnt_mito', 'drug', 'drugname_drugconc']


In [11]:
sm = pd.read_parquet(f"hf://datasets/{REPO}/metadata/sample_metadata.parquet", filesystem=fs)
print(sm["drug"].value_counts().head(10).to_string())

is_dmso = sm["drug"].astype(str).str.contains("DMSO", case=False)
print("\nDMSO wells per plate:")
print(sm[is_dmso].groupby("plate")["sample"].nunique().to_string())
print("\nwells per plate:")
print(sm.groupby("plate")["sample"].nunique().to_string())
print("\nDMSO wells:")
print(sm[is_dmso][["plate", "sample", "drugname_drugconc", "mean_tscp_count"]].to_string(index=False))

drug
DMSO_TF                        28
Adagrasib                      18
Afatinib                        9
Everolimus                      6
Gemcitabine                     6
Regorafenib                     6
Binimetinib                     6
Trametinib                      6
Berbamine (dihydrochloride)     6
Daidzin                         6

DMSO wells per plate:
plate
plate1     2
plate10    2
plate11    2
plate12    2
plate13    2
plate14    2
plate2     2
plate3     2
plate4     2
plate5     2
plate6     2
plate7     3
plate8     3
plate9     3

wells per plate:
plate
plate1     96
plate10    96
plate11    96
plate12    96
plate13    96
plate14    96
plate2     96
plate3     96
plate4     96
plate5     96
plate6     96
plate7     96
plate8     96
plate9     96

DMSO wells:
  plate   sample                              drugname_drugconc  mean_tscp_count
 plate1 smp_1589                       [('DMSO_TF', 0.0, 'uM')]      2401.468362
 plate1 smp_1590                       [('DMSO_TF

In [13]:
print([f for f in files if f.startswith("metadata/")])
with fs.open(f"datasets/{REPO}/{data_files[0]}") as fh:
    pf = pq.ParquetFile(fh)
    print(pf.schema)
    t = pf.read_row_group(0).slice(0, 3).to_pandas()
print(t.drop(columns=[c for c in ["genes", "expressions"] if c in t.columns]).to_string())
print(t[["genes", "expressions"]].map(lambda x: list(x[:8]) if hasattr(x, "__len__") else x).to_string())

['metadata/cell_line_metadata.parquet', 'metadata/drug_metadata.parquet', 'metadata/gene_metadata.parquet', 'metadata/gene_vocabulary.json', 'metadata/gene_vocabulary.jsonl', 'metadata/obs_metadata.parquet', 'metadata/pseudobulk_differential_expression/train-00000-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00001-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00002-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00003-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00004-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00005-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00006-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00007-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00008-of-01026.parquet', 'metadata/pseudobulk_differential_expression/train-00009-of-01026.parquet', 'metadata/pseudobulk_differentia